# Output Alignment Checker &nbsp;`v1`

a test to check the 3 folders

In [0]:
########################
# test Setup
########################

state_under_test = "archive_output_alignment"
run_tag_default  = "output_alignment"

dbutils.widgets.text("run_tag", run_tag_default)
dbutils.widgets.text("sample_size", "5")

run_tag  = dbutils.widgets.get("run_tag") or run_tag_default
SAMPLE_N = int(dbutils.widgets.get("sample_size") or 5)

FS_SEGMENTS = [
    ("FPA",   "ARIADM/ARM/APPEALS/ARIAFPA", "appeals_"),
    ("FTA",   "ARIADM/ARM/APPEALS/ARIAFTA", "appeals_"),
    ("UTA",   "ARIADM/ARM/APPEALS/ARIAUTA", "appeals_"),
    ("TD",    "ARIADM/ARM/TD",              "tribunal_decision_"),
    ("BAIL",  "ARIADM/ARM/BAILS",           "bails_"),
    ("SBAIL", "ARIADM/ARM/SBAILS",          "sbails_"),
]

In [0]:
import os
import sys
import time as perf_time
from datetime import datetime

from pyspark.sql import functions as F

t0 = perf_time.time()
run_start_datetime = datetime.utcnow()

run_user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
test_results_path = "/Workspace/Users/" + run_user + "/Results/Archive_Output_Alignment_Checker"
os.makedirs(test_results_path, exist_ok=True)

for _mod in list(sys.modules.keys()):
    if _mod.startswith("Test_Functions") or _mod == "models.test_result":
        del sys.modules[_mod]

from models.test_result import TestResult
from Test_Functions.report_helpers import (
    build_report_folder,
    write_csv, write_xlsx, write_html,
    count_by_status,
    create_run, create_results, create_perf_results,
)

print(f"User: {run_user}")
print(f"state_under_test: {state_under_test}")
print(f"segments: {[s[0] for s in FS_SEGMENTS]}")
print(f"Results folder root: {test_results_path}")

In [0]:
#######################
# Spark Config
#######################
config = spark.read.option("multiline", "true").json("dbfs:/configs/config.json")
first_row = config.first()
env = first_row["env"].strip().lower()
lz_key = first_row["lz_key"].strip().lower()
keyvault_name = f"ingest{lz_key}-meta002-{env}"

client_secret = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-SECRET")
tenant_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-TENANT-ID")
client_id     = dbutils.secrets.get(scope=keyvault_name, key="SERVICE-PRINCIPLE-CLIENT-ID")

for storage in (f"ingest{lz_key}curated{env}", f"ingest{lz_key}xcutting{env}",
                f"ingest{lz_key}raw{env}", f"ingest{lz_key}landing{env}", f"ingest{lz_key}external{env}"):
    spark.conf.set(f"fs.azure.account.auth.type.{storage}.dfs.core.windows.net", "OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage}.dfs.core.windows.net", client_id)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage}.dfs.core.windows.net", client_secret)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")

print(f"env={env}  lz_key={lz_key}")

## Helpers

`GOLD_CONTAINER` is the gold ADLS container (built from `env` / `lz_key` read in the Spark Config cell).
File listing is done with Spark (`binaryFile`, path column only) so it lists huge folders (e.g. TD's
~2.3M files) **in parallel across the cluster without reading file content** — far faster than `dbutils.fs.ls`.

- `list_stems(folder, ext, prefix)` → a DataFrame of per-case keys derived from the filenames (`/` → `_`).
- `a360_case_keys(folder)` → reads the `.a360` batch files as text and pulls the per-case key from each
  `create_record` line's `relation_id`. The A360 format is 3 JSON lines per case
  (`create_record` + 2× `upload_new_file`), so one `create_record` = one case, and the same key in two
  batches = a duplicate.

In [0]:
import re

GOLD_CONTAINER = f"abfss://gold@ingest{lz_key}curated{env}.dfs.core.windows.net"

def _stem(path_col, ext, prefix):
    fname = F.element_at(F.split(path_col, "/"), -1)
    s = F.regexp_replace(fname, "(?i)" + re.escape(ext) + "$", "")
    return F.regexp_replace(s, "^" + re.escape(prefix), "")

def list_stems(folder, ext, prefix):
    try:
        df = (spark.read.format("binaryFile")
                .option("pathGlobFilter", "*" + ext)
                .option("recursiveFileLookup", "true")
                .load(folder).select("path"))
    except Exception:
        return None
    return df.select(_stem(F.col("path"), ext, prefix).alias("stem"))

def a360_case_keys(folder):
    try:
        txt = (spark.read.format("text")
                .option("pathGlobFilter", "*.a360")
                .option("recursiveFileLookup", "true")
                .load(folder))
    except Exception:
        return None
    return (txt.select(
                F.get_json_object("value", "$.operation").alias("op"),
                F.regexp_replace(F.get_json_object("value", "$.relation_id"), "/", "_").alias("stem"))
            .where("op = 'create_record' and stem is not null")
            .select("stem"))

def _samples_df(df, n=None):
    n = n or SAMPLE_N
    return ", ".join([r[0] for r in df.limit(n).collect()])

## Run the checks

One pass per segment. It lists the actual **JSON**, **HTML** and **A360** output folders on the gold
container and gathers the stats, then runs four checks:

1. **stats** — `json_files`, `html_files`, `a360_batches`, `a360_cases` (distinct cases inside the batch files).
2. **json vs html** — every case has both a `.json` and a `.html` file.
3. **json vs a360** — every JSON case is referenced in the A360 content (`json-not-a360 > 0` is the FTA-style loss).
4. **a360 duplicates** — each case appears in exactly one A360 batch (no `relation_id` in two batches).

In [0]:
all_test_results = []
perf_timings = []

def add(field, status, message, desc=""):
    all_test_results.append(TestResult(
        test_name="output_alignment", test_field=field,
        test_from_state=state_under_test, status=status, message=message, description=desc))

for label, base, prefix in FS_SEGMENTS:
    seg_t0 = perf_time.time()

    json_folder = GOLD_CONTAINER + "/" + base + "/JSON"
    html_folder = GOLD_CONTAINER + "/" + base + "/HTML"
    a360_folder = GOLD_CONTAINER + "/" + base + "/A360"

    js = list_stems(json_folder, ".json", prefix)
    hs = list_stems(html_folder, ".html", prefix)
    ab = list_stems(a360_folder, ".a360", prefix)
    ac = a360_case_keys(a360_folder)

    missing = [n for n, d in [("JSON", js), ("HTML", hs), ("A360", ab), ("A360-content", ac)] if d is None]
    if missing:
        add(label + ":folders", "NO_DATA",
            "Not found: " + str(missing) + ". Checked " + json_folder + " / " + html_folder + " / " + a360_folder,
            "Cannot list one or more output folders — path or permissions issue.")
        perf_timings.append({"test_from_state": label, "test_name": "output_alignment",
                             "elapsed_seconds": perf_time.time() - seg_t0, "result_count": 1})
        continue

    js = js.distinct().cache()
    hs = hs.distinct().cache()
    ac_all = ac.cache()
    ac_distinct = ac_all.distinct().cache()

    jn = js.count()
    hn = hs.count()
    a360_batches = ab.count()
    a360_records = ac_all.count()
    a360_cases = ac_distinct.count()
    a360_dupes = a360_records - a360_cases

    jnh = js.subtract(hs).count()
    hnj = hs.subtract(js).count()
    jna = js.subtract(ac_distinct).count()
    anj = ac_distinct.subtract(js).count()

    add(label + ":stats", "PASS",
        f"json_files={jn} html_files={hn} a360_batches={a360_batches} a360_cases={a360_cases} (a360_records={a360_records})",
        f"File counts in {json_folder} / {html_folder} / {a360_folder}. a360_cases = distinct create_record relation_ids inside the .a360 batch files.")

    add(label + ":json_vs_html", "PASS" if (jnh == 0 and hnj == 0) else "FAIL",
        f"json-not-html={jnh} html-not-json={hnj}",
        "Every case should have both a JSON and an HTML output file.")

    add(label + ":json_vs_a360", "PASS" if (jna == 0 and anj == 0) else "FAIL",
        f"json-not-a360={jna} a360-not-json={anj}",
        "Every JSON case should be referenced in the A360 content (and vice versa). json-not-a360 > 0 is the FTA-style silent loss.")

    add(label + ":a360_duplicates", "PASS" if a360_dupes == 0 else "FAIL",
        f"a360 duplicate case-records={a360_dupes} (records={a360_records}, distinct cases={a360_cases})",
        "Each case must appear in exactly one A360 batch — a relation_id in two batches is a duplicate.")

    if jna > 0:
        add(label + ":json_present_a360_missing", "FAIL",
            f"{jna} cases have a JSON file but are NOT in any A360 batch. Examples: [{_samples_df(js.subtract(ac_distinct))}]",
            f"Present in {json_folder} but absent from the A360 content — dropped from gold inner-join.")
    if a360_dupes > 0:
        dup_df = ac_all.groupBy("stem").count().where("count > 1").select("stem")
        add(label + ":a360_duplicate_examples", "FAIL",
            f"{a360_dupes} duplicate a360 case-records across batches. Example cases: [{_samples_df(dup_df)}]",
            "These cases appear in more than one .a360 batch file.")

    perf_timings.append({"test_from_state": label, "test_name": "output_alignment",
                         "elapsed_seconds": perf_time.time() - seg_t0,
                         "result_count": sum(1 for r in all_test_results if r.test_field.startswith(label + ":"))})

    print(f"{label}: json={jn} html={hn} a360_cases={a360_cases} dupes={a360_dupes} | jnh={jnh} jna={jna}")

## Results

Compute the headline counts, append the `00_OVERALL_SUMMARY` row, sort (summary first, then FAIL),
and write the CSV/XLSX/HTML plus the audit rows to `test_automation_runs2` / `_results2` / `_perfresults2`.

In [0]:
elapsed = perf_time.time() - t0
counts = count_by_status(all_test_results)

print(f"Tests complete in {elapsed:.1f}s")
print(f"  Total : {counts['TOTAL']}")
print(f"  Pass  : {counts['PASS']}")
print(f"  Fail  : {counts['FAIL']}")
print(f"  NoData: {counts['NO_DATA']}")
print(f"  Error : {counts['ERROR']}")

overall_status = "FAIL" if (counts["FAIL"] + counts["ERROR"]) > 0 else "PASS"
all_test_results.append(TestResult(
    test_name="00_OVERALL_SUMMARY",
    test_field="00_OVERALL",
    test_from_state=state_under_test,
    status=overall_status,
    message=(f"Total={counts['TOTAL']} Pass={counts['PASS']} "
             f"Fail={counts['FAIL']} Error={counts['ERROR']} "
             f"NoData={counts['NO_DATA']} Elapsed={elapsed:.1f}s"),
    description="Output file alignment across the JSON/HTML/A360 output folders for all archive segments."))

status_order = {"FAIL": 0, "ERROR": 1, "NO_DATA": 2, "PASS": 3}
def _sort_key(r):
    if str(getattr(r, "test_name", "")) == "00_OVERALL_SUMMARY":
        return (-1, "")
    return (status_order.get(str(r.status).upper(), 4), str(r.test_field))
all_test_results.sort(key=_sort_key)

folder, timestamp, _ = build_report_folder(test_results_path, f"{state_under_test}_{run_tag}")
print(f"Reports folder: {folder}")

try:
    csv_path = write_csv(all_test_results, state_under_test, folder, timestamp)
    print(f"CSV  : {csv_path}")
    write_xlsx(all_test_results, state_under_test, folder, timestamp)
    write_html(all_test_results, state_under_test, folder, timestamp, counts)
except Exception as e:
    print(f"file write FAILED: {e}")

try:
    run_id = create_run(
        spark,
        run_user=run_user,
        run_start_datetime=run_start_datetime,
        run_by_automation_name="output_alignment_checker",
        run_tag=run_tag,
        run_status=("FAILED" if (counts["FAIL"] + counts["ERROR"]) > 0 else "PASSED"),
        state_under_test=state_under_test,
    )
    print(f"Created run_id={run_id}")
    n = create_results(spark, run_id, all_test_results)
    print(f"Wrote {n} result rows")
    p = create_perf_results(spark, run_id, perf_timings, state_under_test)
    print(f"Wrote {p} perf rows")
except Exception as e:
    print(f"audit write FAILED: {e}")

## Final summary

Display the full result table (headline + per-segment alignment + sample-bearing detail rows).

In [0]:
import pandas as pd
pdf = pd.DataFrame(
    [{"test_field": r.test_field, "status": r.status, "message": r.message,
      "test_from_state": r.test_from_state, "test_name": r.test_name} for r in all_test_results])
try:
    display(pdf)
except Exception:
    print(pdf.to_string(index=False))